# ctrl vs stochMCSP ensemble member 0 duplication

Doing some analysis, I noticed very similar numbers for ctrl vs stochMCSP analysis. On further checking, I found the data was identical. How could this be?

Provenance:
* ctrl: u-di727.
* stochMCSP: u-dg135.
* vanillaMCSP: u-di728.
* Both run on Monsoon. scp'd to JASMIN.
* converted from .pp to .nc using iris (~/projects/mcs_prime/ctrl/stoch_trig_remakefiles/convert_pp_to_nc.py)
* extracted precip and combined using (~/projects/mcs_prime/ctrl/stoch_trig_remakefiles/N216_sims_process.py:N216ExtractCombinePrecip)

## Resolution:

* EM0 *is* identical between ctrl and stochMCSP.
* The reason is that stochastics is disabled in EM0, and stochMCSP makes use of the pattern to get probabilities. So it is never activated.

Here's a diff between EM0 and EM1 for stochMCSP on Monsoon:

```
mamue@xcslc0 /projects/mcsprime/mamue/cylc-run/u-di727/work/20200701T0000Z $ diff engl_um_fcst_em0_cr0/SHARED engl_um_fcst_em1_cr0/SHARED                                                                                                     
2,5c2,5                                                                                                                                                                                                                                       
< astart='/home/d02/mamue/cylc-run/u-di727/share/cycle/20200701T0000Z/engl/ics/em0/engl_astart',
< atmanl='/home/d02/mamue/cylc-run/u-di727/share/cycle/20200701T0000Z/engl/ics/em0/engl_astart',
< iau_inc='/home/d02/mamue/cylc-run/u-di727/work/20200701T0000Z/engl_um_fcst_em0_cr0/IAU_incs/engl_um/IAU01',
< streqlog='/home/d02/mamue/cylc-run/u-di727/share/cycle/20200701T0000Z/engl/um/em0/engla.stash',
---
> astart='/home/d02/mamue/cylc-run/u-di727/share/cycle/20200701T0000Z/engl/ics/em1/engl_astart',
> atmanl='/home/d02/mamue/cylc-run/u-di727/share/cycle/20200701T0000Z/engl/ics/em1/engl_astart',
> iau_inc='/home/d02/mamue/cylc-run/u-di727/work/20200701T0000Z/engl_um_fcst_em1_cr0/IAU_incs/engl_um/IAU01',
> streqlog='/home/d02/mamue/cylc-run/u-di727/share/cycle/20200701T0000Z/engl/um/em1/engla.stash',
656a657,660
> br=0.0200,
> cdispfac=1.00,
> conv_std=1.73,
> gwd_std=1.5,
659,660c663,678
< l_skeb2=.false.,
< l_spt=.false.,
---
> l_skeb2=.true.,
> l_skeb2_conv_disp_mod=.true.,
> l_skeb2_psicdisp=.true.,
> l_skeb2_psisdisp=.true.,
> l_skeb2_velpot=.true.,
> l_skebprint=.true.,
> l_skebsmooth_adv=.true.,
> l_spt=.true.,
> l_spt_cfl=.true.,
> l_spt_conv=.true.,
> l_spt_conv_mom=.false.,
> l_spt_gwd=.true.,
> l_spt_mse=.true.,
> l_spt_qcons=.true.,
> l_spt_rad=.true.,
> l_spt_rain=.true.,
661a680,695
> nsmooth=3,
> nsmooth_spt=3,
> psif_orog_thres=0.5,
> rad_std=1.73,
> rain_std=1.73,
> sd_orog_thres=500.,
> sdispfac=2.00,
> skeb2_botlev=2,
> skeb2_cdisp=4,
> skeb2_sdisp=7,
> skeb2_toplev=50,
> spt_bot_tap_lev=9,
> spt_botlev=15,
> spt_top_tap_lev=45,
> spt_toplev=41,
> stph_n1=20,
663a698,700
> tau_skeb=2.00e+4,
> tau_spt=2.00e+4,
> tot_backscat=1.00e-4,
```

In [ ]:
from hashlib import sha1
from pathlib import Path

import xarray as xr

In [2]:
basepath = Path('/gws/nopw/j04/mcs_prime/mmuetz/data/UM_sims')

In [3]:
stochMCSPpath = basepath / 'u-dg135/processed/stochMCSP/engla_pa.precip.nc'

In [6]:
da_stoch = xr.load_dataarray(stochMCSPpath)

In [5]:
ctrlpath = basepath / 'u-di727/processed/ctrl/engla_pa.precip.nc'

In [7]:
da_ctrl = xr.load_dataarray(ctrlpath)

In [10]:
# Shows that ONLY em0 has the same values.
(da_stoch.values == da_ctrl.values).all(axis=(1, 2, 3))

array([ True, False, False, False, False, False, False, False, False,
       False])

In [13]:
for axis in [(0, 1, 2), (0, 1, 3), (0, 2, 3), (1, 2, 3)]:
    print(axis, (da_stoch.values == da_ctrl.values).all(axis=axis).any())

(0, 1, 2) False
(0, 1, 3) False
(0, 2, 3) False
(1, 2, 3) True


In [14]:
da_stoch

<xarray.DataArray 'precipitation_flux' (ens_mem: 10, time: 240, latitude: 324,
                                        longitude: 432)>
array([[[[1.58357234e-11, 1.59586667e-11, 1.62573531e-11, ...,
          1.57854198e-11, 1.55686783e-11, 1.56609777e-11],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         ...,
         [2.00361959e-04, 2.01866264e-04, 2.03914984e-04, ...,
          1.96447174e-04, 1.98376656e-04, 1.98968221e-04],
         [2.78846826e-04, 2.84102367e-04, 2.89610238e-04, ...,
          2.69149721e-04, 2.72195146e-04, 2.76357750e-04],
         [2.16374683e-04, 2.15517939e-04, 2.15388354e-04, ...,
          2.17069959e-04, 2.17442561e-04, 2.17493172e-04]],

        [[1.52534235e-11, 1.50900108e-11, 1.45477085e-11, ...,
          1.61220412e-11, 1.57194032e-11, 1.52608169e-11],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
...
          9.81131780e-06, 9.76066349e-06, 9.89450109e-06],
         [4.58682334e-05, 4.84667980e-05, 5.82353969e-05, ...,
          3.86632782e-05, 4.00932513e-05, 3.99725759e-05],
         [1.14363902e-04, 1.18485994e-04, 1.21341873e-04, ...,
          1.07742489e-04, 1.12091133e-04, 1.14818875e-04]],

        [[6.46807996e-10, 6.21736218e-10, 6.04723549e-10, ...,
          6.59934940e-10, 6.74711509e-10, 6.60473287e-10],
         [1.76322490e-09, 1.77217696e-09, 1.78510795e-09, ...,
          1.77756110e-09, 1.76635018e-09, 1.76055348e-09],
         [5.22891996e-09, 5.18244203e-09, 5.13249665e-09, ...,
          5.35245359e-09, 5.31577671e-09, 5.27350785e-09],
         ...,
         [1.12028147e-05, 1.38273253e-05, 1.42714671e-05, ...,
          7.87960835e-06, 8.18584249e-06, 1.01319492e-05],
         [3.96092000e-05, 4.28365820e-05, 4.72053216e-05, ...,
          3.96398937e-05, 3.90753085e-05, 3.55509001e-05],
         [1.57016970e-04, 1.59650153e-04, 1.64472382e-04, ...,
          1.50548483e-04, 1.56738941e-04, 1.54109162e-04]]]],
      dtype=float32)
Coordinates:
  * latitude   (latitude) float32 -89.72 -89.17 -88.61 ... 88.61 89.17 89.72
  * longitude  (longitude) float32 0.4167 1.25 2.083 2.917 ... 357.9 358.7 359.6
  * time       (time) datetime64[ns] 2020-07-01T04:00:00 ... 2020-07-11T03:00:00
  * ens_mem    (ens_mem) int64 0 1 2 3 4 5 6 7 8 9
Attributes:
    standard_name:        precipitation_flux
    units:                kg m-2 s-1
    um_stash_source:      m01s05i216
    grid_mapping:         latitude_longitude
    UM simulation:        u-dg135
    MCS:PRIME expt:       stochMCSP
    created by:           /home/users/mmuetz/projects/mcs_prime/ctrl/stoch_tr...
    calling file source:  import numpy as np\nimport pandas as pd\nimport xar...
    project repository:   https://github.com/markmuetz/MCS_PRIME
    remake version:       ['0', '7', '0', '0', 'beta']
    remake repository:    https://github.com/markmuetz/remake
    task:                 <N216_sims_process.N216ExtractCombinePrecip object ...
    task doc:             Extract and combine precipitation at all times and ...
    created on:           2024-09-03 14:50:54.092847
    nodename:             host649.jc.rl.ac.uk
    hostname:             jasmin
    output path:          /gws/nopw/j04/mcs_prime/mmuetz/data/UM_sims/u-dg135...
    contact:              mark.muetzelfeldt@reading.ac.uk
    coordinates:          forecast_period_0 forecast_reference_time_0

In [23]:
ds_t0_stoch = xr.open_dataset(basepath / 'u-dg135/share/cycle/20200701T0000Z/engl/um/em0/englaa_pa000.iris.nc')
ds_t0_ctrl = xr.open_dataset(basepath / 'u-di727/share/cycle/20200701T0000Z/engl/um/em0/englaa_pa000.iris.nc')

In [24]:
(ds_t0_stoch.precipitation_flux.values == ds_t0_ctrl.precipitation_flux.values).all()

True

In [25]:
ds_t0_stoch

<xarray.Dataset>
Dimensions:                                                                                (
                                                                                            time: 25,
                                                                                            latitude: 324,
                                                                                            longitude: 432,
                                                                                            time_0: 24,
                                                                                            time_1: 24,
                                                                                            bnds: 2,
                                                                                            depth: 4,
                                                                                            time_3: 23,
                                                                                            time_4: 24,
                                                                                            latitude_0: 325,
                                                                                            longitude_0: 432)
Coordinates: (12/25)
  * time                                                                                   (time) datetime64[ns] ...
  * latitude                                                                               (latitude) float32 ...
  * longitude                                                                              (longitude) float32 ...
    forecast_period                                                                        (time) timedelta64[ns] ...
    forecast_reference_time                                                                (time) datetime64[ns] ...
  * time_0                                                                                 (time_0) datetime64[ns] ...
    ...                                                                                     ...
    forecast_period_4                                                                      (time_4) timedelta64[ns] ...
    forecast_reference_time_3                                                              (time_4) datetime64[ns] ...
    height_0                                                                               float32 ...
  * latitude_0                                                                             (latitude_0) float32 ...
  * longitude_0                                                                            (longitude_0) float32 ...
    height_1                                                                               float64 ...
Dimensions without coordinates: bnds
Data variables: (12/47)
    m01s09i202                                                                             (time, latitude, longitude) float32 ...
    latitude_longitude                                                                     int32 ...
    cloud_area_fraction_assuming_only_consider_surface_to_1000_feet_asl                    (time, latitude, longitude) float32 ...
    m01s09i233                                                                             (time, latitude, longitude) float32 ...
    m01s01i202                                                                             (time_0, latitude, longitude) float32 ...
    turbulent_mixing_height_after_boundary_layer                                           (time_0, latitude, longitude) float32 ...
    ...                                                                                     ...
    toa_incoming_shortwave_flux                                                            (time_0, latitude, longitude) float32 ...
    toa_outgoing_longwave_flux                                                             (time_4, latitude, longitude) float32 ...
    toa_outgoing_shortwave_flux                          

In [19]:
import iris

In [26]:
cubes_t0_stoch = iris.load(basepath / 'u-dg135/share/cycle/20200701T0000Z/engl/um/em0/englaa_pa000.pp')
cubes_t0_ctrl = iris.load(basepath / 'u-di727/share/cycle/20200701T0000Z/engl/um/em0/englaa_pa000.pp')


In [30]:
cubes_t0_stoch

M01S09I202 (unknown),time,latitude,longitude
Shape,25,324,432
Dimension coordinates,,,
time,x,-,-
latitude,-,x,-
longitude,-,-,x
Auxiliary coordinates,,,
forecast_period,x,-,-
forecast_reference_time,x,-,-
Attributes,,,STASH m01s09i202source 'Data from Met Office Unified Model'um_version '13.0'
Cloud Area Fraction Assuming Only Consider Surface To 1000 Feet Asl (1),time,latitude,longitude


In [31]:
cubes_t0_ctrl

M01S09I202 (unknown),time,latitude,longitude
Shape,25,324,432
Dimension coordinates,,,
time,x,-,-
latitude,-,x,-
longitude,-,-,x
Auxiliary coordinates,,,
forecast_period,x,-,-
forecast_reference_time,x,-,-
Attributes,,,STASH m01s09i202source 'Data from Met Office Unified Model'um_version '13.0'
Cloud Area Fraction Assuming Only Consider Surface To 1000 Feet Asl (1),time,latitude,longitude


In [33]:
(cubes_t0_stoch[24].data == cubes_t0_ctrl[23].data).all()

True

OK, so the .pp files have the same issue where em0 is duplicated by ctrl and stochMCSP. Let's go to Monsoon:

# Monsoon

On Monsoon, the rose suites have the correct setup:

```
mamue@xcslc0 ~/roses $ diff u-{dg135,di728}/app/engl_um/rose-app.conf
2020c2020
< l_org_stoch_trigger=.true.
---
> l_org_stoch_trigger=.false.
mamue@xcslc0 ~/roses $ diff u-{dg135,di727}/app/engl_um/rose-app.conf
2019,2020c2019,2020
< l_org_conv=.true.
< l_org_stoch_trigger=.true.
---
> l_org_conv=.false.
> l_org_stoch_trigger=.false.
3076,3083d3075
<
< [namelist:umstash_streq(2)]
< dom_name='STD_DIAG'
< isec=5
< item=993
< package='STD_DIAGS'
< tim_name='STD_T1HR_MN'
< use_name='STD_DIAG'
```

Furthermore, the suite namelists have the correct setup (em0):

```
mamue@xcslc0 /projects/mcsprime/mamue/cylc-run $ grep -I l_org u-{dg135,di727,di728}/work/20200701T0000Z/engl_um_fcst_em0_cr0/SHARED                                                                                                         
u-dg135/work/20200701T0000Z/engl_um_fcst_em0_cr0/SHARED:l_org_conv=.true.,
u-dg135/work/20200701T0000Z/engl_um_fcst_em0_cr0/SHARED:l_org_stoch_trigger=.true.,
u-di727/work/20200701T0000Z/engl_um_fcst_em0_cr0/SHARED:l_org_conv=.false.,
u-di727/work/20200701T0000Z/engl_um_fcst_em0_cr0/SHARED:l_org_stoch_trigger=.false.,
u-di728/work/20200701T0000Z/engl_um_fcst_em0_cr0/SHARED:l_org_conv=.true.,
u-di728/work/20200701T0000Z/engl_um_fcst_em0_cr0/SHARED:l_org_stoch_trigger=.false.,
```

And the proc output for each suite is good (em0):

```
mamue@xcslc0 /projects/mcsprime/mamue/cylc-run $ grep -I l_org u-{dg135,di727,di728}/work/20200701T0000Z/engl_um_fcst_em0_cr0/pe_output.1/engla.fort6.pe00                                                                                   
u-dg135/work/20200701T0000Z/engl_um_fcst_em0_cr0/pe_output.1/engla.fort6.pe00:cv_run_mod:l_org_conv = T
u-dg135/work/20200701T0000Z/engl_um_fcst_em0_cr0/pe_output.1/engla.fort6.pe00:cv_run_mod:l_org_stoch_trigger = T
u-di727/work/20200701T0000Z/engl_um_fcst_em0_cr0/pe_output.1/engla.fort6.pe00:cv_run_mod:l_org_conv = F
u-di727/work/20200701T0000Z/engl_um_fcst_em0_cr0/pe_output.1/engla.fort6.pe00:cv_run_mod:l_org_stoch_trigger = F
u-di728/work/20200701T0000Z/engl_um_fcst_em0_cr0/pe_output.1/engla.fort6.pe00:cv_run_mod:l_org_conv = T
u-di728/work/20200701T0000Z/engl_um_fcst_em0_cr0/pe_output.1/engla.fort6.pe00:cv_run_mod:l_org_stoch_trigger = F
```

And org_conv is being called for stochMCSP (em0):

```
mamue@xcslc0 /projects/mcsprime/mamue/cylc-run $ grep -I org u-{dg135,di727,di728}/work/20200701T0000Z/engl_um_fcst_em0_cr0/pe_output.1/engla.fort6.pe00                                                                                      
...
u-dg135/work/20200701T0000Z/engl_um_fcst_em0_cr0/pe_output.1/engla.fort6.pe00:org_conv:org_conv called
...
```                                                  


## Summary

The problem stretches back to Monsoon, and the output of each suite. This is despite the suites being setup correctly, and each namelist getting the correct values for l_org_conv and l_org_conv_stoch_trigger. I am at a bit of a loss for what to do now.